In [1]:
import pandas as pd 
import numpy as np 
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

#### Note:
joblib is used for saving python objects to files and loading them later especially useful for ml datasets, trained models and preprocessing objects.

In [2]:
dataset = pd.read_csv(r"D:\Study\Credit-Card-Fraud-Detection-System\data\creditcard.csv")
dataset.head(3)

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0


In [3]:
dataset.shape

(284807, 31)

In [4]:
dataset.drop(columns=['Time'], inplace= True)

In [5]:
scaler = StandardScaler()
dataset['Amount'] = scaler.fit_transform(dataset[['Amount']])
print("Amount is scaled here and the sample values are:\n ", dataset['Amount'].values[:5])

Amount is scaled here and the sample values are:
  [ 0.24496426 -0.34247454  1.16068593  0.14053425 -0.07340334]


In [6]:
joblib.dump(scaler, "../data/scaler.pkl")
print("Scaler saved.")

Scaler saved.


In [7]:
x = dataset.iloc[:,:-1]
y = dataset["Class"]

print("x shape is: ", x.shape)
print("y value count: ", y.value_counts())

x shape is:  (284807, 29)
y value count:  Class
0    284315
1       492
Name: count, dtype: int64


In [8]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2, random_state=42)

In [16]:
y_test[y_test==1].index  # provides all the index having fraud case from test value only

Index([ 43428, 116404, 215984, 249239, 146790,  77099,  15810, 261473, 280149,
        82400,  73857, 204503, 124176, 108258, 189701, 143188, 214775,  44270,
       102442, 229730, 237426,  42635, 150697, 151006, 108708, 235644,  68067,
        64329,  43061, 154454, 143728,  42784, 150654, 213116,   8615, 233258,
       144754, 263324,  11710,  39183, 243749,  41569,  89190,   9252,  12070,
        79883,  94218,  88876,  42007, 163586,  79536, 137705,  20198, 249167,
       154587, 154718, 219025,  68522,  74507, 150663,  77387, 123238,  72757,
        44091, 280143,  84543, 247673,  93486, 226877, 263877, 118308, 154371,
        15781,  17407,  93788,   6882,  16780,  77348, 191359,  15225, 249828,
       116139, 149577, 274475,  81186, 149600,  64460,  11880, 150661, 151730,
       102443, 244333, 111690, 152223,   6903, 276864, 199896, 248971],
      dtype='int64')

In [14]:
dataset.iloc[43428]  # This gives all the features having fraud case

V1       -16.526507
V2         8.584972
V3       -18.649853
V4         9.505594
V5       -13.793819
V6        -2.832404
V7       -16.701694
V8         7.517344
V9        -8.507059
V10      -14.110184
V11        5.299236
V12      -10.834006
V13        1.671120
V14       -9.373859
V15        0.360806
V16       -9.899247
V17      -19.236292
V18       -8.398552
V19        3.101735
V20       -1.514923
V21        1.190739
V22       -1.127670
V23       -2.358579
V24        0.673461
V25       -1.413700
V26       -0.462762
V27       -2.018575
V28       -1.042804
Amount     1.102834
Class      1.000000
Name: 43428, dtype: float64

In [9]:
smote = SMOTE(random_state=42)
x_train_sm, y_train_sm = smote.fit_resample(x_train, y_train)

In [10]:
print("Before SMOTE: ", y_train.value_counts().to_dict())
print("After SMOTE: ", pd.Series(y_train_sm).value_counts().to_dict())

Before SMOTE:  {0: 227451, 1: 394}
After SMOTE:  {0: 227451, 1: 227451}


After smote, y_train_sm becomes numpy array and not pandas series so .value_counts() will fail as it is pandas function. so, pd.Series is used to convert it into series.

### Save preprocessed data

In [11]:
joblib.dump((x_train_sm, x_test, y_train_sm, y_test), '../data/preprocessed_splits.pkl')
print("saved to ../data/preprocessed_splits.pkl")

saved to ../data/preprocessed_splits.pkl


#### Note:
That path is relative it works because your notebook is inside the `notebooks/` folder.
When you run `joblib.dump(..., '../data/preprocessed_splits.pkl')` from inside `notebooks/`, the `..` means "go up one folder", landing you in `fraud-detection/`, then into `data/`.
So the file saves to:
```
fraud-detection/data/preprocessed_splits.pkl
```
where .pkl is pickle file: a binary file storing python objects

## Summary
- Dropped Time column
- Scaled Amount with StandardScaler
- Split 80/20 before SMOTE
- Applied SMOTE only to training data
- Fraud samples in training went from ~383 → ~227,451 (balanced)
- Saved splits to data/preprocessed_splits.pkl for next phase